# Apify + ETL: guía introductoria

**Objetivo:** entender qué es Apify, qué es un pipeline ETL, y cómo Apify encaja como capa de **Extract** en un flujo de datos real, usando como ejemplo el actor de extracción de texto de PDFs (`5Z6nOka1yB74YDPyI`).

> Notebook pensado para correr en Google Colab. Las celdas que requieren un `APIFY_API_TOKEN` real y conexión a `api.apify.com` están protegidas con `try/except` para que el notebook corra completo incluso en modo demo (sin token / sin salida a internet).


## 1. ¿Qué es Apify?

[Apify](https://apify.com) es una plataforma de **web scraping y automatización** que expone "Actors": programas empaquetados (scrapers, crawlers, herramientas de procesamiento) que corren en la infraestructura de Apify y se invocan vía API.

Puntos clave:

- **Actor**: unidad ejecutable (ej: un scraper de Amazon, un extractor de texto de PDFs, un crawler genérico).
- **Run**: una ejecución concreta de un Actor, con su propio `id`, input, logs y estado.
- **Dataset**: donde el Actor escribe sus resultados (una fila = un item, formato JSON).
- **Input**: un JSON que parametriza la corrida (URLs, límites, timeouts, etc).

El actor del ejemplo (`5Z6nOka1yB74YDPyI`) recibe una lista de URLs de PDFs y devuelve el texto extraído de cada página.


## 2. ¿Qué es ETL?

**ETL** = **Extract, Transform, Load**. Es el patrón estándar para mover datos desde una fuente cruda hacia un destino analizable:

| Etapa | Qué hace | Ejemplo típico |
|---|---|---|
| **Extract** | Obtiene datos crudos de una fuente externa | Scrapear una web, leer una API, descargar PDFs |
| **Transform** | Limpia, normaliza, filtra, enriquece | Parsear texto, tipar columnas, deduplicar |
| **Load** | Persiste el resultado en el destino final | CSV, base de datos, data warehouse (Snowflake, etc) |

El orden puede variar (ELT invierte Transform y Load), pero la lógica de separar responsabilidades es la misma.


## 3. ¿Dónde entra Apify en un pipeline ETL?

Apify resuelve casi siempre la etapa de **Extract**, cuando la fuente de datos:

- No tiene una API pública, o
- Requiere navegación/renderizado (JS, paginación, login), o
- Es un archivo no estructurado (como PDFs) que hay que parsear.

Flujo típico:

```
[Apify Actor] --(Extract)--> [Dataset JSON] --(Transform)--> [pandas / script propio] --(Load)--> [CSV / DB / Snowflake]
```

En este notebook vamos a recorrer las tres etapas con el actor de extracción de PDFs como ejemplo.


## 4. Instalación del cliente

Apify tiene SDKs oficiales en **JavaScript** y **Python**. Como este notebook es Python (Colab), usamos `apify-client` en vez del `apify-client` de npm que aparecía en el snippet original en JS.


In [ ]:
%pip install -q apify-client pandas

error: externally-managed-environment

× This environment is externally managed
╰─> To install Python packages system-wide, try apt install
    python3-xyz, where xyz is the package you are trying to
    install.
    
    If you wish to install a non-Debian-packaged Python package,
    create a virtual environment using python3 -m venv path/to/venv.
    Then use path/to/venv/bin/python and path/to/venv/bin/pip. Make
    sure you have python3-full installed.
    
    If you wish to install a non-Debian packaged Python application,
    it may be easiest to use pipx install xyz, which will manage a
    virtual environment for you. Make sure you have pipx installed.
    
    See /usr/share/doc/python3.12/README.venv for more information.

note: If you believe this is a mistake, please contact your Python installation or OS distribution provider. You can override this, at the risk of breaking your Python installation or OS, by passing --break-system-packages.
hint: See PEP 668 for the detai

Note: you may need to restart the kernel to use updated packages.


## 5. Extract — correr el Actor y traer el Dataset

Equivalente en Python del snippet original en JavaScript:

```js
const client = new ApifyClient({ token: '<YOUR_API_TOKEN>' });
const run = await client.actor("5Z6nOka1yB74YDPyI").call(input);
const { items } = await client.dataset(run.defaultDatasetId).listItems();
```

En Python:


In [ ]:
from apify_client import ApifyClient

APIFY_API_TOKEN = "<YOUR_API_TOKEN>"  # reemplazar por tu token real (Settings > Integrations en Apify Console)

client = ApifyClient(APIFY_API_TOKEN)

run_input = {
    "urls": [
        "https://www.w3.org/WAI/ER/tests/xhtml/testfiles/resources/pdf/dummy.pdf",
        "https://www.orimi.com/pdf-test.pdf",
    ],
    "maxConcurrency": 5,
    "timeoutPerPdfSecs": 60,
    "includePages": True,
}

items = []

try:
    # Extract: corre el actor y espera a que termine
    run = client.actor("5Z6nOka1yB74YDPyI").call(run_input=run_input)

    # Trae los resultados del dataset asociado a esa corrida
    dataset_items = client.dataset(run["defaultDatasetId"]).list_items().items
    items = dataset_items
    print(f"Extract OK: {len(items)} items obtenidos del dataset.")

except Exception as e:
    # Modo demo: no hay token válido o no hay salida a api.apify.com en este entorno
    print("Modo demostración: no se pudo conectar a Apify (esperado sin token/conexión real).")
    print(f"Detalle: {e}")
    items = [
        {
            "url": "https://www.w3.org/WAI/ER/tests/xhtml/testfiles/resources/pdf/dummy.pdf",
            "pages": [{"pageNumber": 1, "text": "Dummy PDF file"}],
        },
        {
            "url": "https://www.orimi.com/pdf-test.pdf",
            "pages": [{"pageNumber": 1, "text": "PDF Test file. This is a Test PDF File."}],
        },
    ]

items

Modo demostración: no se pudo conectar a Apify (esperado sin token/conexión real).
Detalle: Unexpected error: Host not in allowlist: api.apify.com. Add this host to your network egress settings to allow access.


[{'url': 'https://www.w3.org/WAI/ER/tests/xhtml/testfiles/resources/pdf/dummy.pdf',
  'pages': [{'pageNumber': 1, 'text': 'Dummy PDF file'}]},
 {'url': 'https://www.orimi.com/pdf-test.pdf',
  'pages': [{'pageNumber': 1,
    'text': 'PDF Test file. This is a Test PDF File.'}]}]

## 6. Transform — normalizar el resultado

El dataset de Apify devuelve JSON anidado (una URL con una lista de páginas). Para análisis conviene aplanarlo a una tabla: una fila por página.


In [ ]:
import pandas as pd

rows = []
for item in items:
    url = item.get("url")
    for page in item.get("pages", []):
        rows.append({
            "url": url,
            "page_number": page.get("pageNumber"),
            "text": page.get("text", "").strip(),
            "char_count": len(page.get("text", "")),
        })

df = pd.DataFrame(rows)
df

,url,page_number,text,char_count
0,https://www.w3.org/WAI/ER/tests/xhtml/testfile...,1,Dummy PDF file,14
1,https://www.orimi.com/pdf-test.pdf,1,PDF Test file. This is a Test PDF File.,39


## 7. Load — persistir el resultado

Último paso: guardar el DataFrame ya transformado. Acá lo mandamos a CSV local; en un pipeline real este paso sería un `INSERT`/`COPY INTO` a Snowflake, Postgres, BigQuery, etc.


In [ ]:
output_path = "pdf_extract_output.csv"
df.to_csv(output_path, index=False)
print(f"Load OK: {len(df)} filas guardadas en {output_path}")

Load OK: 2 filas guardadas en pdf_extract_output.csv


## 8. Resumen del pipeline

| Etapa | Herramienta usada | Resultado |
|---|---|---|
| Extract | Apify Actor (`5Z6nOka1yB74YDPyI`) vía `apify-client` | JSON crudo por URL/página |
| Transform | `pandas` | Tabla plana, una fila por página |
| Load | `DataFrame.to_csv` | `pdf_extract_output.csv` |

### Para ir más allá
- Reemplazar el CSV final por una conexión real (Snowflake, Postgres) en el paso de Load.
- Encadenar varios Actors (ej: un crawler que descubre URLs de PDFs + este actor que las procesa).
- Agregar reintentos / backoff en el Extract para runs largas.
